## Simulating Ibtesam Ahmed recommendation algorithms from Kaggle

### Content-based recommender doesn't use ML, Collaborative does

#### Link: https://www.kaggle.com/code/ibtesama/getting-started-with-a-movie-recommendation-system/notebook

# Neural Content-Based Recommender
## Plot + Credits, Genres & Keywords via Sentence Transformers

In [1]:
import pandas as pd
import numpy as np

df1=pd.read_csv('../tmdb/tmdb_5000_credits.csv')
df2=pd.read_csv('../tmdb/tmdb_5000_movies.csv')

In [ ]:
# Join two datasets on id column
df1.columns = ['id','title','cast','crew']
df2= df2.merge(df1,on='id')

In [ ]:
df2.head()

In [4]:
df2['overview'].head(5)

0    In the 22nd century, a paraplegic Marine is di...
1    Captain Barbossa, long believed to be dead, ha...
2    A cryptic message from Bond’s past sends him o...
3    Following the death of District Attorney Harve...
4    John Carter is a war-weary, former military ca...
Name: overview, dtype: str

In [5]:
# Parse the stringified features into their corresponding python objects
from ast import literal_eval

features = ['cast', 'crew', 'keywords', 'genres']
for feature in features:
    df2[feature] = df2[feature].apply(literal_eval)

#### Functions that will help extract required info from each feature

In [6]:
# Get the director's name from the crew feature. If director is not listed, return NaN
def get_director(x):
    for i in x:
        if i['job'] == 'Director':
            return i['name']
    return np.nan

In [10]:
# Test directors function

df2['director'] = df2['crew'].apply(get_director)

df2[['title', 'director']].head()

,title,director
0,Avatar,James Cameron
1,Pirates of the Caribbean: At World's End,Gore Verbinski
2,Spectre,Sam Mendes
3,The Dark Knight Rises,Christopher Nolan
4,John Carter,Andrew Stanton


In [14]:
# Returns the list top 3 elements or entire list; whichever is more.
def get_list(x):
    if isinstance(x, list):
        names = [i['name'] for i in x]
        
        # Check if more than 3 elements exist. If yes, return only first three. If no, return entire list.
        if len(names) > 3:
            names = names[:3]
        return names

    # Return empty list in case of missing/malformed data
    return []

In [17]:
# Define new director, cast, genres and keywords features that are in a suitable form.
df2['director'] = df2['crew'].apply(get_director)

features = ['cast', 'keywords', 'genres']
for feature in features:
    df2[feature] = df2[feature].apply(get_list)

TypeError: string indices must be integers, not 'str'

In [16]:
# Print the new features of the first 3 films
df2[['title', 'cast', 'director', 'keywords', 'genres']].head(3)

,title,cast,director,keywords,genres
0,Avatar,"[Sam Worthington, Zoe Saldana, Sigourney Weave...",James Cameron,"[culture clash, future, space war, space colon...","[Action, Adventure, Fantasy, Science Fiction]"
1,Pirates of the Caribbean: At World's End,"[Johnny Depp, Orlando Bloom, Keira Knightley, ...",Gore Verbinski,"[ocean, drug abuse, exotic island, east india ...","[Adventure, Fantasy, Action]"
2,Spectre,"[Daniel Craig, Christoph Waltz, Léa Seydoux, R...",Sam Mendes,"[spy, based on novel, secret agent, sequel, mi6]","[Action, Adventure, Crime]"


#### The next step would be to convert the names and keyword instances into lowercase and strip all the spaces between them. This is done so that our vectorizer doesn't count the Johnny of "Johnny Depp" and "Johnny Galecki" as the same.

In [18]:
# Function to convert all strings to lower case and strip names of spaces
def clean_data(x):
    if isinstance(x, list):
        return [str.lower(i.replace(" ", "")) for i in x]
    else:
        # Check if director exists. If not, return empty string
        if isinstance(x, str):
            return str.lower(x.replace(" ", ""))
        else:
            return ''

In [19]:
# Apply clean_data function to your features.
features = ['cast', 'keywords', 'director', 'genres']

for feature in features:
    df2[feature] = df2[feature].apply(clean_data)

## Step 1: Build a unified text representation for each movie
#### We combine the plot overview with structured metadata (cast, director, keywords, genres) into one rich text string per movie. Metadata terms are repeated to give them comparable signal weight alongside the longer plot text.

In [20]:
def create_unified_text(x):
    plot     = x['overview']              if isinstance(x['overview'],  str) else ''
    cast     = ' '.join(x['cast'])        if isinstance(x['cast'],     list) else ''
    genres   = ' '.join(x['genres'])      if isinstance(x['genres'],   list) else ''
    keywords = ' '.join(x['keywords'])    if isinstance(x['keywords'], list) else ''
    director = x['director']              if isinstance(x['director'],  str) else ''
    
    # Metadata terms repeated twice to balance against the longer plot text
    return f"{plot} {cast} {cast} {director} {director} {genres} {genres} {keywords} {keywords}".strip()

df2['overview'] = df2['overview'].fillna('')
df2['unified_text'] = df2.apply(create_unified_text, axis=1)
df2['unified_text'].head(3)

0    In the 22nd century, a paraplegic Marine is di...
1    Captain Barbossa, long believed to be dead, ha...
2    A cryptic message from Bond’s past sends him o...
Name: unified_text, dtype: str

## Step 2: Generate neural embeddings with Sentence Transformers
#### The `all-MiniLM-L6-v2` model is a lightweight BERT-based neural network trained to produce semantically meaningful 384-dimensional vectors. Unlike TF-IDF or CountVectorizer, it understands meaning — so "hero saves the world" and "protagonist rescues humanity" will score as similar even with zero word overlap.

In [21]:
%pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 20.7 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 618.0/618.0 kB 17.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 39.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 34.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 802.0/802.0 kB 29.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 22.3 MB/s  0:00:12m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 32.0 MB/s  0:00:08m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 48.8 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 47.9 MB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 52.5 MB/s  0:00:016m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 49.3 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 12.7 MB/

In [22]:
from sentence_transformers import SentenceTransformer

# Load pre-trained neural model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode all movies into 384-dimensional neural embedding vectors
# show_progress_bar gives visibility since this encodes ~4800 movies
embeddings = model.encode(
    df2['unified_text'].tolist(),
    show_progress_bar=True,
    batch_size=64
)

print(f'Embeddings shape: {embeddings.shape}')

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6282.82it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 76/76 [04:17<00:00,  3.39s/it]

Embeddings shape: (4803, 384)


## Step 3: Compute cosine similarity between all movie embeddings
#### Each movie is now a point in 384-dimensional neural space. Cosine similarity measures the angle between two points — movies with similar meaning cluster together regardless of exact word overlap.

In [23]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute pairwise cosine similarity across all neural embeddings
cosine_sim_nn = cosine_similarity(embeddings, embeddings)

# Build reverse index: movie title -> dataframe index
indices = pd.Series(df2.index, index=df2['title']).drop_duplicates()

print(f'Similarity matrix shape: {cosine_sim_nn.shape}')

Similarity matrix shape: (4803, 4803)


## Step 4: Recommendation function
#### Takes a movie title and returns the 10 most similar movies based on the neural similarity matrix.

In [35]:
def get_recommendations(title):
    
    # Make title all lowsercase
    title = title.lower()

    # Get copy of indices and make them lowercase to compare to title
    indices_lower = indices.copy()
    indices_lower.index = indices_lower.index.str.lower()

    # Return message if movie is not in list
    if title not in indices_lower:
        return f"Movie '{title}' not found in database."

    # get index of title in indices
    idx = indices_lower[title]

    # Get similarity scores for this movie against all others
    sim_scores = list(enumerate(cosine_sim_nn[idx]))
    
    # Sort by score descending
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Skip index 0 (the movie itself), and print top 10
    sim_scores = sim_scores[1:11]

    movie_indices = [i[0] for i in sim_scores]
    return df2['title'].iloc[movie_indices]

In [41]:
# Year range

df2['release_date'] = pd.to_datetime(df2['release_date'], errors='coerce')
df2['year'] = df2['release_date'].dt.year

min_year = df2['year'].min()
max_year = df2['year'].max()

print(min_year, max_year)

1916.0 2017.0


## Step 5: Test the recommender

#### Test users and movies they enjoy

In [53]:
user_history = {
    # User 1: Classic gangster & crime dramas
    "user_1": [
        "The Godfather", "GoodFellas", "Scarface", "Pulp Fiction", "The Departed",
        "The Godfather: Part II", "Casino", "Donnie Brasco", "Once Upon a Time in America",
        "The Untouchables", "Road to Perdition", "Public Enemies", "Gangs of New York",
        "A History of Violence", "Eastern Promises", "The Town", "The Conformist",
        "Find Me Guilty", "Black Mass", "The Hills Have Eyes"
    ],

    # User 2: Blockbuster action & superhero movies
    "user_2": [
        "Avatar", "Titanic", "Avengers: Age of Ultron", "Guardians of the Galaxy", "Iron Man",
        "Thor", "Captain America: The First Avenger", "The Avengers", "Ant-Man",
        "The Incredible Hulk", "Captain America: Civil War", "Iron Man 2", "Iron Man 3",
        "Thor: The Dark World", "Batman Begins", "The Dark Knight", "The Dark Knight Rises",
        "Batman & Robin", "Batman Returns", "Batman v Superman: Dawn of Justice"
    ],

    # User 3: Musical & biographical movies
    "user_3": [
        "Chicago", "Moulin Rouge!", "8MM", "Amnesiac", "Grease",
        "Les Misérables", "Inception", "The Pursuit of Happyness", "The Hit List",
        "Singin' in the Rain", "The Sound of Music", "West Side Story", "Mary Poppins",
        "The Wizard of Oz", "Frozen", "Aladdin", "Cinderella", "The Nutcracker",
        "Alice in Wonderland", "The Broadway Melody"
    ],

    # User 4: Fantasy & young adult series
    "user_4": [
        "The Lord of the Rings: The Fellowship of the Ring", "The Hobbit: An Unexpected Journey",
        "The Lord of the Rings: The Two Towers", "The Lord of the Rings: The Return of the King",
        "Harry Potter and the Philosopher's Stone", "Harry Potter and the Chamber of Secrets",
        "Harry Potter and the Prisoner of Azkaban", "The Hunger Games: Catching Fire",
        "The Hunger Games: Mockingjay - Part 2", "The Twilight Saga: New Moon",
        "The Twilight Saga: Eclipse", "The Twilight Saga: Breaking Dawn - Part 2",
        "Percy Jackson: Sea of Monsters", "Percy Jackson & the Olympians: The Lightning Thief",
        "Harry Potter and the Order of the Phoenix", "The Chronicles of Narnia: The Lion, the Witch and the Wardrobe",
        "The Hobbit: The Desolation of Smaug", "The Hobbit: The Battle of the Five Armies",
        "The Adventures of Huck Finn", "Hellboy II: The Golden Army"
    ],

    # User 5: Horror & thriller movies
    "user_5": [
        "The Shining", "1408", "8 Days", "The Conjuring", "Insidious",
        "Sinister", "Annabelle", "Paranormal Activity 2", "Halloween: Resurrection", "Psycho",
        "Jaws", "Saw: The Final Chapter", "Scream 3", "Pet Sematary", "White Noise 2: The Light",
        "It Follows", "The Possession", "The Exorcist", "Evil Dead", "Restoration"
    ]
}

In [ ]:
# Test get_recommendations for every liked movie of each user

counter = 0

for user, movies in user_history.items():
    print(f"\nRecommendations for {user}:")
    for movie in movies:
        #print(f"\nMovie: {movie}")
        recs = get_recommendations(movie)
        if "not found" in recs:
            counter +=1
            print(recs)
            
print(f"\nTotal movies not found: {counter}")

In [55]:
import random

In [ ]:
# Split movies into training and testing to evaluate model
train_test_split = {}
split_ratio = 5

for user, movies in user_history.items():
    known = random.sample(movies, split_ratio)  # movies user has "rated" / known
    unknown = [m for m in movies if m not in known]  # remaining movies for evaluation
    train_test_split[user] = {"known": known, "unknown": unknown}

# Example output:
for user, split in train_test_split.items():
    print(f"{user}:")
    print("Known:", split["known"])
    print("Unknown:", split["unknown"])
    print()

In [ ]:
# Generate Recommendations

# Feed known movies into recommender

# Check how many movies in the unknown list appear in the recommendation (hit rate)

In [ ]:
# Evaluate

# Precision@K: How many recommended movies were actually liked (in unknown set).
# Recall@k: How many of the unknown liked movies did the model manage to recommend.

In [ ]:
get_recommendations('The Dark Knight Rises')

In [ ]:
get_recommendations('The Godfather')

In [ ]:
get_recommendations('The Avengers')